# Arena GPU Bridge — Kaggle × Colab 算力桥

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaohuawen03-cyber/rea/blob/arena/cac2205a-rea/cloud-gpu/colab/Arena_GPU_Bridge.ipynb)

**用法(3 步):**
1. 菜单「代码执行程序 → 更改运行时类型」→ 硬件加速器选 **GPU**(T4/L4/A100 视额度);
2. 从上到下「全部运行」;
3. 跑完把最后一格的 `OUTBOX` 路径或结果贴回 Arena 会话,agent 会去 Google Drive 读取。

**任务来源:** agent 通过 google_drive 连接器把任务放在 `MyDrive/arena-bridge/inbox/`
(`job.json` + `job.py`);没有任务时自动跑内置 GPU 冒烟。结果统一写
`MyDrive/arena-bridge/outbox/<job-id>/`。

> 打不开徽章?访问 https://colab.research.google.com → 文件 → 上传笔记本 →
> 选仓库里的 `cloud-gpu/colab/Arena_GPU_Bridge.ipynb`。

In [ ]:
# ============ 单元格 1:环境自检 ============
import os, platform, subprocess, sys

print("python:", sys.version.split()[0], "|", platform.platform())
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "[no nvidia-smi]")

try:
    import torch
    print("torch:", torch.__version__, "| cuda_available:", torch.cuda.is_available(),
          "| devices:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"  cuda:{i}", torch.cuda.get_device_name(i))
except Exception as e:
    print("torch not ready:", e)
print("[env check done]")

In [ ]:
# ============ 单元格 2:挂载 Google Drive(算力桥信箱) ============
BRIDGE_ROOT = "/content/drive/MyDrive/arena-bridge"
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(BRIDGE_ROOT + "/inbox", exist_ok=True)
    os.makedirs(BRIDGE_ROOT + "/outbox", exist_ok=True)
    DRIVE_MOUNTED = True
    print("Drive mounted. bridge root:", BRIDGE_ROOT)
except Exception as e:
    print("[Drive mount skipped]", e)
    BRIDGE_ROOT = "/content/arena-bridge"
    os.makedirs(BRIDGE_ROOT + "/inbox", exist_ok=True)
    os.makedirs(BRIDGE_ROOT + "/outbox", exist_ok=True)
    print("fallback local bridge root:", BRIDGE_ROOT)

In [ ]:
# ============ 单元格 3:读取 inbox 任务(没有就用内置冒烟) ============
import json, pathlib

inbox = pathlib.Path(BRIDGE_ROOT) / "inbox"
job_json = inbox / "job.json"
job_py = inbox / "job.py"

if job_json.exists():
    job = json.loads(job_json.read_text(encoding="utf-8"))
    print("inbox job found:", json.dumps(job, ensure_ascii=False))
else:
    job = {"id": "colab-smoke", "script": "builtin", "args": []}
    print("no inbox job; built-in GPU smoke will run")

JOB_ID = job.get("id", "colab-smoke")
RUN_DIR = pathlib.Path(BRIDGE_ROOT) / "outbox" / JOB_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============ 单元格 4:执行任务 ============
import io, time, traceback

t0 = time.time()
stdout_buf = io.StringIO()
result = {"job_id": JOB_ID, "status": "unknown"}

def _builtin_smoke():
    import torch
    assert torch.cuda.is_available(), "no CUDA GPU — 请在「更改运行时类型」里选 GPU"
    x = torch.randn(4096, 4096, device="cuda")
    y = (x @ x).float().mean().item()
    props = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    return {"gpu_names": props, "matmul_mean": y,
            "torch": torch.__version__, "cuda": torch.version.cuda}

try:
    if job.get("script") == "builtin" or not job_py.exists():
        metrics = _builtin_smoke()
    else:
        import runpy, shlex
        sys_argv_backup = sys.argv
        sys.argv = [str(job_py), *[str(a) for a in job.get("args", [])]]
        try:
            runpy.run_path(str(job_py), run_name="__main__")
        finally:
            sys.argv = sys_argv_backup
        metrics = {"note": f"{job_py.name} executed", "see": "stdout.log"}
    result = {"job_id": JOB_ID, "status": "completed", "metrics": metrics,
              "wall_sec": round(time.time() - t0, 2)}
except Exception:
    result = {"job_id": JOB_ID, "status": "failed",
              "error": traceback.format_exc()[-2000:],
              "wall_sec": round(time.time() - t0, 2)}

result["finished_at_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# ============ 单元格 5:结果写入 outbox ============
import json, pathlib, sys, time

run_dir = pathlib.Path(RUN_DIR)
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
try:
    import torch
    env_info = {"torch": torch.__version__, "cuda_available": torch.cuda.is_available()}
except Exception:
    env_info = {}
(run_dir / "env.json").write_text(json.dumps(env_info, indent=2) + "\n", encoding="utf-8")

# inbox 任务是一次性的:执行后移入 done/ 防止重复运行
if DRIVE_MOUNTED:
    done = pathlib.Path(BRIDGE_ROOT) / "inbox" / "done"
    done.mkdir(exist_ok=True)
    for f in ("job.json", "job.py"):
        p = pathlib.Path(BRIDGE_ROOT) / "inbox" / f
        if p.exists():
            p.rename(done / (f + "." + str(int(time.time()))))

print("OUTBOX =", run_dir)
print("[done] 把上面的 OUTBOX 路径或结果贴回 Arena 会话即可")